<a href="https://colab.research.google.com/github/johnmichaels4867-create/ACCTG5150/blob/main/labs_hw/week2_connecting-to-data/Lab%202%20-%20Opening%20and%20Cleaning%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 - Opening and Cleaning Data

<a href="https://colab.research.google.com/github/gaulinmp/AccountingDataAnalytics/blob/main/labs_hw/week2_connecting-to-data/Lab 2 - Opening and Cleaning Data.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook will walk you through the process of opening and cleaning the Journal Entry dataset. We will cover the following steps:

1. Loading the data
2. Cleaning the data

The intent of the notebook is for you to execute each cell in order. Some will work, others will throw errors, and then I'll explain what's going on, and help you through it.
If you're new to programming, it can sometimes feel daunting, so I want to provide these initial notebooks to help you get started, and hopefully show you that programming (especially in Colab) isn't all that bad once you try it out.

In [3]:
# This is a comment

# The code below are called imports, which is basically telling python to load specific functionality
# This one uses a python library called pathlib to access the Path object, which is just a way of pointing to files on the computer
from pathlib import Path
# This import is the real magic, pandas is the library that makes data manipulation easy (well... easier)
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Point to the file, and make sure it exists
file_path = Path("/content/drive/MyDrive/ACCT_5150/JEA Detail Raw.txt")
file_path.exists()

True

That should say True above. If it says False, then your data is not loaded correctly. Please make sure to drag the `JAE Detail Raw.txt` file into your Colab page (see Lab 1 for instructions).

Okay, now to actually load the data:

In [5]:
# Load the data into pandas
df = pd.read_csv(file_path)
# Display the first few rows of the DataFrame
df.head()

ParserError: Error tokenizing data. C error: Expected 1 fields in line 14, saw 2


Oh no! An Error!

Welcome to debugging. This is often the hardest and least rewarding part of programming, because while there is often a few right ways to do something, there are infinite *wrong* ways to do something. So it's like searching for a needle in an infinite ocean of rusty needles.

*Future Mac here: I wrote this lab on my local machine, then tried it out on Google Colab. Colab shows you a button that says Next Steps: Explain Error. If you click this button, it immediately comes up with the solution. Being suspicious, I tried a new notebook with just the pandas read, and it took me 6 or 7 leading questions with this button and interacting with Gemini before I got the full solution working. So in this notebook, Gemini is reading down below, sees that I've solved the problem, and passes off the solution as its own. AI is all just plagerism anyway, so maybe unsurprising? Anyway, if you want the "authentic" coding with AI in Colab experience, I suggest starting an empty notebook and writing the code from scratch with its help. That'll be more indicative of what you might expect in practice. And now I know that Google Colab is great for debugging much faster than with just my meat brain!*

In this case, we look at the error, specifically `ParserError` and try googling for that. So I copy and paste that last line into [Gemini](https://gemini.google.com) (or [Claude](https://claude.ai) or [ChatGPT](https://chat.openai.com)), which tells me:

<blockquote>
  This is a common **Pandas** error. It usually occurs when `read_csv` tries to parse a file but finds a row that violates the structure established by the header or the first row.

Specifically, "Expected 1 fields... saw 2" usually implies one of two things:

1. **Wrong Separator:** Pandas failed to find the default separator (comma) in the first row, so it assumed the file has only 1 column. Then, on line 14, it found a character that acted like a separator.
2. **Metadata/Junk at the top:** The file has a title or description in the first few lines that looks like a single column, confusing the parser before it hits the actual data.

Here are the most effective ways to fix this.

...
</blockquote>

It then suggests to try a different separater, starting with tab:

In [6]:
# Load the data into pandas
df = pd.read_csv(file_path, sep='\t')
# Display the first few rows of the DataFrame
df.head()

ParserError: Error tokenizing data. C error: Expected 10 fields in line 3, saw 37


Same error, different numbers. What's happening now? Well the error seems to say it expected 10 fields, but saw 37.

Okay, let's look at the data (we should have started here, but we're eager) (also note that exclamation mark `!head` command in the next line, that's not Python. Then what *is* it? I suggest you ask the internet to find out this lovely little bit of Notebook tech).

In [9]:
!head "{file_path}"

Company XYZ 									 PAGE 0001
REPORT DATE: 02/07/22  12:32
																																				
			GENERAL LEDGER DETAIL
			AS OF 12/31/21
																																				
																																				
Account	"Account
Category"	"Post


Uh... that's ugly. What's going on? Well, a few things. Playing with raw data is hard, Excel, Tableau, Alteryx, etc. have put a lot of code into trying to figure out things like this. But in Python, we'd have to do it ourselves. So we can see that there's a bunch of text that isn't the actual file we want. In fact, we can't even see the data we expect. Let look at more lines:

In [10]:
!head -n 20 "{file_path}"

Company XYZ 									 PAGE 0001
REPORT DATE: 02/07/22  12:32
																																				
			GENERAL LEDGER DETAIL
			AS OF 12/31/21
																																				
																																				
Account	"Account
Category"	"Post
Date"	Period	"User
ID"	Manual	Description	"Transaction
Number"	Location	Amount				
9504 Rent Expense	SG&A	9/11/2024	9	Laura4	Yes	Rent	1000	LA	"50,000.00"
9504 Rent Expense	SG&A	8/11/2024	8	"
AUTOMATED
"	No	Rent	1001	LA	"50,000.00"
9504 Rent Expense	SG&A	7/11/2024	7	BeanCounter25	Yes	Rent	1002	LA	"50,000.00"
9504 Rent Expense	SG&A	6/11/2024	6	BeanCounter25	Yes	Rent	1003	LA	"50,000.00"
9504 Rent Expense	SG&A	5/11/2024	5	BeanCounter25	Yes	Rent	1004	LA	"50,000.00"


Well there's the data, finally. So we're closer. But there's a bunch of random text above the table we want, and those headers (line with `Account "Account`) are ugly and splits the column names across two lines. That means that we need to *a)* skip the first 8 lines (you can count to verify), and *b)* strip newlines from the column names.

In [11]:
# Load the data into pandas
df = pd.read_csv(file_path, sep='\t', thousands=',', skiprows=8)
# Rename the columns, using a trick to remove extra spaces and newlines. Essentially, we split each column name on whitespace, then re-join with a single space, and strip any leading/trailing spaces.
df = df.rename(columns={c:' '.join(c.split()).strip() for c in df.columns})
# Display the first few rows of the DataFrame
df.head(10)

,Account,Account Category,Post Date,Period,User ID,Manual,Description,Transaction Number,Location,Amount,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,9504 Rent Expense,SG&A,9/11/2024,9.0,Laura4,Yes,Rent,1000.0,LA,"50,000.00",NaN,NaN,NaN,NaN
1,9504 Rent Expense,SG&A,8/11/2024,8.0,\r\nAUTOMATED\r\n,No,Rent,1001.0,LA,"50,000.00",NaN,NaN,NaN,NaN
2,9504 Rent Expense,SG&A,7/11/2024,7.0,BeanCounter25,Yes,Rent,1002.0,LA,"50,000.00",NaN,NaN,NaN,NaN
3,9504 Rent Expense,SG&A,6/11/2024,6.0,BeanCounter25,Yes,Rent,1003.0,LA,"50,000.00",NaN,NaN,NaN,NaN
4,9504 Rent Expense,SG&A,5/11/2024,5.0,BeanCounter25,Yes,Rent,1004.0,LA,"50,000.00",NaN,NaN,NaN,NaN
5,9504 Rent Expense,SG&A,4/11/2024,4.0,\r\nAUTOMATED\r\n,No,Rent,1005.0,LA,"50,000.00",NaN,NaN,NaN,NaN
6,9504 Rent Expense,SG&A,3/11/2024,3.0,\r\nAUTOMATED\r\n,No,Rent,1006.0,LA,"50,000.00",NaN,NaN,NaN,NaN
7,9504 Rent Expense,SG&A,2/11/2024,2.0,\r\nAUTOMATED\r\n,No,Rent,1007.0,LA,"50,000.00",NaN,NaN,NaN,NaN
8,9504 Rent Expense,SG&A,1/11/2024,1.0,Laura4,Yes,Rent,1008.0,LA,"50,000.00",NaN,NaN,NaN,NaN
9,9504 Rent Expense,SG&A,10/11/2024,10.0,\r\nAUTOMATED\r\n,No,Rent,1009.0,LA,"50,000.00",NaN,NaN,NaN,NaN


Now let's drop the three columns at the end that are entirely empty. That's what the `how='all'` means. `axis=1` means check columns, `axis=0` would check the rows for all missings.

In [12]:
df = df.dropna(how='all', axis=1)

Now let's remove newlines (`\n`) from the user names. While you're at it, [look up](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.html) what `.str` is doing.

In [13]:
df["User ID"] = df["User ID"].str.strip()

Let's look at the cleaned DataFrame again (I'll cheat and show you a few rows down, so you can see the next issue, which is what [`iloc`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iloc.html) is doing):

In [14]:
df.iloc[42:48]

,Account,Account Category,Post Date,Period,User ID,Manual,Description,Transaction Number,Location,Amount
42,9504 Rent Expense,SG&A,7/25/2024,7.0,BeanCounter25,Yes,Rent,1042.0,SF,"20,000.00"
43,9504 Rent Expense,SG&A,8/25/2024,8.0,Karen15,Yes,Rent,1043.0,SF,"20,000.00"
44,9504 Rent Expense,SG&A,9/25/2024,9.0,AUTOMATED,No,Rent,1044.0,SF,"20,000.00"
45,==============================================...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PAGE 0002
47,9504 Rent Expense,SG&A,10/25/2024,10.0,AUTOMATED,No,Rent,1045.0,SF,"20,000.00"


So we have some bad lines to remove. I suggest dropping them by requiring one of the columns to be non-missing.

In [15]:
df = df.dropna(subset=['User ID'])

Lastly, we have "data types" to fix. If you look at the `dtypes` attribute, you'll see that the `Amount` and `Post Date` columns are `object`, which means they are strings, not a date or number as we want them to be.

In [16]:
df.dtypes

,0
Account,object
Account Category,object
Post Date,object
Period,float64
User ID,object
Manual,object
Description,object
Transaction Number,float64
Location,object
Amount,object


But we know that these are actually a numeric value and date respectively, so we you should convert them to the appropriate types.

*Hint:* there are two functions you can use: `pd.to_datetime()` for dates and `pd.to_numeric()` for numeric values. Removing quotes and commas can be done with string manipulation: `df['Amount'].str.replace("[\",]", "", regex=True)`.

In [18]:
df['Post Date'] = pd.to_datetime(df['Post Date'])
df['Amount'] = pd.to_numeric(df['Amount'].str.replace("[\[\],]", "", regex=True))

<>:2: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_949/3884551837.py:2: SyntaxWarning: invalid escape sequence '\['
  df['Amount'] = pd.to_numeric(df['Amount'].str.replace("[\[\],]", "", regex=True))


Now, the last step is to produce the Lab's output, which is a screenshot of the cleaned table, and a pivot table counting how many journal entries each `User ID` has. There are a few ways to do this, and if you put your cursor after `df.pivot_table(` and wait, Gemini might even suggest a solution for you (but only because I wrote that comment, if it weren't there, then it wouldn't know what you want. Remember this for future reference, it's *incredibly useful* to know that you can often just write a comment and get AI to give you the code to do it).

In [24]:
df.head(10)

,Account,Account Category,Post Date,Period,User ID,Manual,Description,Transaction Number,Location,Amount
0,9504 Rent Expense,SG&A,2024-09-11,9.0,Laura4,Yes,Rent,1000.0,LA,50000.0
1,9504 Rent Expense,SG&A,2024-08-11,8.0,AUTOMATED,No,Rent,1001.0,LA,50000.0
2,9504 Rent Expense,SG&A,2024-07-11,7.0,BeanCounter25,Yes,Rent,1002.0,LA,50000.0
3,9504 Rent Expense,SG&A,2024-06-11,6.0,BeanCounter25,Yes,Rent,1003.0,LA,50000.0
4,9504 Rent Expense,SG&A,2024-05-11,5.0,BeanCounter25,Yes,Rent,1004.0,LA,50000.0
5,9504 Rent Expense,SG&A,2024-04-11,4.0,AUTOMATED,No,Rent,1005.0,LA,50000.0
6,9504 Rent Expense,SG&A,2024-03-11,3.0,AUTOMATED,No,Rent,1006.0,LA,50000.0
7,9504 Rent Expense,SG&A,2024-02-11,2.0,AUTOMATED,No,Rent,1007.0,LA,50000.0
8,9504 Rent Expense,SG&A,2024-01-11,1.0,Laura4,Yes,Rent,1008.0,LA,50000.0
9,9504 Rent Expense,SG&A,2024-10-11,10.0,AUTOMATED,No,Rent,1009.0,LA,50000.0


In [20]:
df.pivot_table(
    index='User ID',
    values='Transaction Number',
    aggfunc='count',
    margins=True,
    margins_name="Total",
)

,Transaction Number
User ID,
AUTOMATED,23495
BeanCounter25,117
Bob2,76
CFO2,25
Karen15,70
Kayla,2
Laura4,78
Total,23863


In [21]:
# For the record, the pivot_table function's arguments are:

df.pivot_table(
    index='Row Variable',             # What you want to group by and see in rows, e.g., User ID
    columns='Column Variable',        # What you want your different calculations to be in columns, e.g., 'Period'
    values='Value to be aggregated',  # What value you actually want to be calculated in each cell, e.g., 'Transaction Number'
    aggfunc='how to aggregate',       # How you want to process those values, e.g., 'mean' (default), 'sum', 'count', 'median', 'std', etc.
    margins=True,                     # Add a total column
    margins_name="Total",             # Set the total column name
)

KeyError: 'Value to be aggregated'

In [26]:
# What is the sum total of all amounts in the data set?
total_amount = df['Amount'].sum()
print(f"The sum of all amounts is: {total_amount:.2f}")

The sum of all amounts is: 0.00


Now take a screenshot, and you're done! I'd save this notebook, because Homework 2 builds on it.

In [27]:
# What is the sum of all Amounts associated with the user "Kayla"?

kayla_amount = df[df['User ID'] == 'Kayla']['Amount'].sum()
print(f"The sum of all Amounts associated with the user Kayla is: {kayla_amount:.2f}")


The sum of all Amounts associated with the user Kayla is: -15000.00


In [28]:
# How many journal entries are in the data set?

num_entries = len(df)
print(f"The total number of journal entries in the data set is: {num_entries}")

The total number of journal entries in the data set is: 23863


In [29]:
# How many unique transactions are represented in the dataset?

unique_transactions = df['Transaction Number'].nunique()
print(f"The total number of unique transactions in the dataset is: {unique_transactions}")

The total number of unique transactions in the dataset is: 6422


In [33]:
# Order the User IDs by the number of journal entries they are associated with, from most to fewest.

user_entry_counts = df['User ID'].value_counts()
display(user_entry_counts)


,count
User ID,
AUTOMATED,23495
BeanCounter25,117
Laura4,78
Bob2,76
Karen15,70
CFO2,25
Kayla,2


In [35]:
# There are 18 Accounts with only one associated Journal Entry. Upload a screenshot of the cleaned dataset, filtered to just show those 18 Journal Entries.

account_counts = df['Account'].value_counts()
single_entry_accounts = account_counts[account_counts == 1].index

# Filter the original DataFrame to show only these single-entry journal entries
filtered_df = df[df['Account'].isin(single_entry_accounts)]
display(filtered_df)

,Account,Account Category,Post Date,Period,User ID,Manual,Description,Transaction Number,Location,Amount
287,8001 Common Stock,Common Stock,2024-05-15,5.0,BeanCounter25,Yes,Stock Issuance,1319.0,LA,-1.000000e+05
288,8000 Additional paid in capital,Additional paid in capital,2024-05-15,5.0,BeanCounter25,Yes,Stock Issuance,1319.0,LA,-1.000000e+07
592,7200 Interest Payable,Interest Payable,2024-12-31,12.0,AUTOMATED,No,Bond Interest,1334.0,LA,-7.500000e+04
597,3000 Prepaid Insurance,Prepaid Expenses,2024-12-31,12.0,BeanCounter25,Yes,Prepaid allocation,1334.0,LA,2.500000e+04
598,3001 Prepaid Legal,Prepaid Expenses,2024-12-31,12.0,BeanCounter25,Yes,Prepaid allocation,1335.0,LA,4.000000e+04
599,3002 Prepaid Rent,Prepaid Expenses,2024-12-31,12.0,BeanCounter25,Yes,Prepaid allocation,1336.0,LA,1.000000e+05
627,6000 Building,PP&E,2024-03-01,3.0,Laura4,Yes,Building purchase,1326.0,LA,4.000000e+06
628,6001 Weaving Equipment,PP&E,2024-04-20,4.0,Laura4,Yes,Equipment purchase,1351.0,LA,2.000000e+04
629,6002 Dying Equipment,PP&E,2024-04-25,4.0,Laura4,Yes,Equipment sale,1352.0,LA,-5.000000e+03
630,6003 Land,PP&E,2024-07-25,7.0,Laura4,Yes,Land Purchase,1353.0,LA,1.000000e+06
